# PyTorch Dataset and DataLoader

**PyTorch Interview Preparation · Notebook 07 · Estimated study time: 2–3 hours**

This notebook builds the bridge between stored samples and a training loop. Everything is synthetic, offline, CPU-safe, and deliberately small.

> **Critical mental model**
>
> `Dataset` → defines one sample  
> `DataLoader` → builds batches from samples  
> training loop → consumes batches

Repeatedly: build a Dataset → retrieve samples → create a DataLoader → inspect batches → predict shapes → customize collation → feed a model → debug.

## 1. Setup

Only PyTorch is required. `num_workers=0` is used in runnable cells for reliable execution across Linux, Windows, macOS, and Jupyter.

In [ ]:
import math
import torch
import torch.nn as nn

from torch.utils.data import (
    Dataset,
    TensorDataset,
    DataLoader,
    random_split,
    WeightedRandomSampler,
)
from torch.nn.utils.rnn import pad_sequence

torch.manual_seed(42)
print("PyTorch version:", torch.__version__)

## 2. What Is a Dataset? 🟢 Easy

### 🔥 Interview Essential

**A PyTorch Dataset represents a collection of samples and defines how individual samples are retrieved.**

Its map-style contract answers two questions:

1. How many samples exist? → `__len__()`
2. How do I retrieve sample `i`? → `__getitem__(i)`

Thus `dataset[0]`, `dataset[1]`, … each retrieve one sample. A Dataset does not train a model or choose a batch size.

## 3. What Is a DataLoader? 🟢 Easy

### 🔥 Interview Essential

**A DataLoader takes a Dataset and provides convenient iteration over batches of samples.**

```text
Dataset
   ↓
DataLoader  (batching, shuffling, multiprocessing, collation)
   ↓
Batch 1 → Batch 2 → Batch 3 → …
```

Compatible tensors are automatically stacked along a new leading batch dimension.

## 4. Dataset vs DataLoader 🔥 Very Common Interview Question

| Dataset | DataLoader |
|---|---|
| represents samples | produces batches |
| defines retrieval of one sample | iterates over many samples |
| implements `__len__` and `__getitem__` | handles batching, shuffling, workers, and collation |
| use `dataset[i]` | use `for batch in loader` |

**Short interview answer:** Dataset defines how individual samples are accessed; DataLoader wraps the Dataset and handles batching, shuffling, parallel loading, and iteration.

## 5. TensorDataset 🟢 Easy

`TensorDataset` is ideal when all data is already stored in aligned tensors. Every tensor must have the same first-dimension length because dimension 0 is the sample dimension.

In [ ]:
X = torch.randn(100, 10)
y = torch.randint(0, 3, (100,))
dataset = TensorDataset(X, y)

sample_X, sample_y = dataset[0]
print("samples:", len(dataset))
print("one feature shape:", sample_X.shape, "label:", sample_y)

assert len(dataset) == 100
assert sample_X.shape == (10,)
assert sample_y.ndim == 0

`X.shape == (100, 10)` and `y.shape == (100,)`: both contain 100 samples. `TensorDataset(torch.randn(100, 10), torch.zeros(90))` fails because the leading lengths disagree.

In [ ]:
# Exercise 1 — create a TensorDataset from aligned tensors.
features = torch.randn(200, 5)
targets = torch.randint(0, 2, (200,))
exercise_dataset = TensorDataset(features, targets)

assert isinstance(exercise_dataset, TensorDataset)
assert len(exercise_dataset) == 200
x0, y0 = exercise_dataset[0]
assert x0.shape == (5,) and y0.ndim == 0
print("✅ TensorDataset exercise correct")

In [ ]:
# Debug the mismatch safely: catch and inspect the expected error.
try:
    TensorDataset(torch.randn(100, 10), torch.zeros(90))
except AssertionError:
    print("Expected failure: tensors do not have equal sample counts.")

## 6. DataLoader Basics and Iteration 🟢 Easy

### 🔥 Interview Essential: `batch_size`, batching

- `dataset`: source of individual samples
- `batch_size`: samples combined per iteration
- `shuffle`: whether index order is randomized

In [ ]:
dataset = TensorDataset(torch.randn(100, 10), torch.randint(0, 3, (100,)))
loader = DataLoader(dataset, batch_size=32, shuffle=True, num_workers=0)

X_batch, y_batch = next(iter(loader))
print("feature batch:", X_batch.shape)
print("label batch:", y_batch.shape)

assert X_batch.shape == (32, 10)
assert y_batch.shape == (32,)

In [ ]:
# Exercise 2 — build a loader with batch_size=16 and shuffle=True.
from torch.utils.data import RandomSampler

loader_exercise = DataLoader(exercise_dataset, batch_size=16, shuffle=True)
assert isinstance(loader_exercise, DataLoader)
assert loader_exercise.batch_size == 16
assert isinstance(loader_exercise.sampler, RandomSampler)
print("✅ DataLoader exercise correct")

## 7. Batch Size and Number of Batches 🟢 Easy

**Batch size is the number of samples processed together in one training step.**

Smaller batches use less memory, create noisier gradient estimates, and produce more optimizer steps per epoch. Larger batches use more memory, produce fewer steps, and may use hardware more efficiently. Neither is universally better.

With `drop_last=False`, `len(loader) = ceil(dataset_size / batch_size)`. For 100 samples and batch size 32, batches are `32, 32, 32, 4`, so `len(loader) == 4`. For 1000 and 128, it is 8.

In [ ]:
count_dataset = TensorDataset(torch.randn(100, 5), torch.randn(100))
count_loader = DataLoader(count_dataset, batch_size=32)
batch_sizes = [xb.shape[0] for xb, _ in count_loader]
print("len(loader):", len(count_loader), "batch sizes:", batch_sizes)
assert len(count_loader) == 4
assert batch_sizes == [32, 32, 32, 4]

## 8. Batch Shape Exercises (15) 🟢 Easy

Predict first; then open the answers.

| # | Data / full shape | Batch size | Normal full-batch shape |
|---:|---|---:|---|
| 1 | tabular features `(1000, 20)` | 64 | ||
| 2 | RGB images `(5000, 3, 224, 224)` | 32 | ||
| 3 | fixed token IDs `(900, 128)` | 16 | ||
| 4 | scalar features `(75,)` | 8 | ||
| 5 | grayscale images `(700, 1, 28, 28)` | 50 | ||
| 6 | embeddings `(2048, 768)` | 128 | ||
| 7 | time series `(300, 24, 6)` | 10 | ||
| 8 | segmentation masks `(120, 256, 256)` | 4 | ||
| 9 | audio features `(640, 80, 100)` | 20 | ||
| 10 | binary labels `(1000,)` | 64 | ||
| 11 | one-hot labels `(1000, 10)` | 64 | ||
| 12 | video clips `(90, 8, 3, 64, 64)` | 3 | ||
| 13 | 3-D points `(400, 1024, 3)` | 25 | ||
| 14 | language features `(250, 32, 512)` | 5 | ||
| 15 | regression targets `(999, 2)` | 100 | ||

<details><summary>Batch-shape answers</summary>

1. `(64, 20)`
2. `(32, 3, 224, 224)`
3. `(16, 128)`
4. `(8,)`
5. `(50, 1, 28, 28)`
6. `(128, 768)`
7. `(10, 24, 6)`
8. `(4, 256, 256)`
9. `(20, 80, 100)`
10. `(64,)`
11. `(64, 10)`
12. `(3, 8, 3, 64, 64)`
13. `(25, 1024, 3)`
14. `(5, 32, 512)`
15. `(100, 2)`

The leading sample dimension is replaced by the current batch size. The final batch can be smaller.
</details>

## 9. Final Incomplete Batch and `drop_last` 🟢 Easy

### 🔥 Interview Essential

By default the final smaller batch is retained. `drop_last=True` discards it. Use dropping only when equal batch shapes are required, a tiny batch causes BatchNorm issues, or a particular training setup needs it. It discards samples each epoch, so it is usually undesirable for validation/test evaluation.

In [ ]:
tiny_dataset = TensorDataset(torch.arange(10))
keep_loader = DataLoader(tiny_dataset, batch_size=4, shuffle=False)
drop_loader = DataLoader(tiny_dataset, batch_size=4, shuffle=False, drop_last=True)

kept = [batch[0].tolist() for batch in keep_loader]
dropped = [batch[0].tolist() for batch in drop_loader]
print("drop_last=False:", kept)
print("drop_last=True: ", dropped)
assert [len(x) for x in kept] == [4, 4, 2]
assert [len(x) for x in dropped] == [4, 4]

## 10. Shuffling 🟢 Easy

### 🔥 Interview Essential: `shuffle=True`

Shuffling changes the sample order each epoch so training does not repeatedly see the same sequence. It does **not** modify samples and is not augmentation.

Typical pattern: training `shuffle=True`; validation `shuffle=False`. Validation parameters are not updated, so order normally does not affect aggregate metrics; stable order also makes debugging and prediction alignment easier.

In [ ]:
order_dataset = TensorDataset(torch.arange(10))
loader_no_shuffle = DataLoader(order_dataset, batch_size=5, shuffle=False)
loader_shuffle = DataLoader(
    order_dataset, batch_size=5, shuffle=True,
    generator=torch.Generator().manual_seed(42)
)

def observed_order(a_loader):
    return torch.cat([batch[0] for batch in a_loader]).tolist()

print("not shuffled:", observed_order(loader_no_shuffle))
print("shuffled:    ", observed_order(loader_shuffle))

**Quick check:** Which changes sample values: shuffling or augmentation?  
<details><summary>Answer</summary>Augmentation may change a sample; shuffling only changes its position.</details>

## 11. Building a Custom Dataset 🟢 Easy

### 🔥 Interview Essential: custom Dataset, `__len__`, `__getitem__`

- `__init__`: store tensors, paths, labels, metadata, transforms, or reusable preprocessing objects.
- `__len__`: return the number of samples; `len(dataset)` calls it.
- `__getitem__(idx)`: return one sample; `dataset[5]` invokes `__getitem__(5)`.

DataLoader repeatedly asks the Dataset for indexed samples and collates the results.

In [ ]:
class ClassificationDataset(Dataset):
    def __init__(self, X, y):
        if len(X) != len(y):
            raise ValueError("X and y must contain the same number of samples")
        self.X = X
        self.y = y

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


custom_X = torch.randn(50, 8)
custom_y = torch.randint(0, 4, (50,))
custom_dataset = ClassificationDataset(custom_X, custom_y)
sample_x, sample_y = custom_dataset[10]

assert len(custom_dataset) == 50
assert sample_x.shape == (8,) and sample_y.ndim == 0
assert torch.equal(sample_x, custom_X[10])
print("✅ Custom Dataset works")

### Write a Custom Dataset From Memory — attempt 1

Before looking above, implement a Dataset accepting `X` and `y` and returning one `(x, y)` pair. The completed reference below validates length, random indices, values, and shape.

In [ ]:
class MyDataset(Dataset):
    def __init__(self, X, y):
        self.X, self.y = X, y

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


memory_X = torch.randn(37, 6)
memory_y = torch.randint(0, 2, (37,))
memory_dataset = MyDataset(memory_X, memory_y)
assert len(memory_dataset) == 37
for idx in torch.randint(0, 37, (5,)).tolist():
    mx, my = memory_dataset[idx]
    assert mx.shape == (6,)
    assert torch.equal(mx, memory_X[idx]) and torch.equal(my, memory_y[idx])
print("✅ Memory implementation validated")

## 12. Preprocessing and Transform Pattern 🟡 Medium

Preprocessing can happen in `__getitem__`, but dataset-wide statistics or expensive reusable setup should not be recomputed for every sample. `__getitem__` may run many times and in several worker processes.

Avoid repeatedly loading huge resources, initializing tokenizers/models, or recomputing global statistics. Store reusable state once where appropriate.

In [ ]:
class NormalizedDataset(Dataset):
    def __init__(self, X, y, transform=None):
        self.X, self.y = X, y
        self.mean = X.mean(dim=0)
        self.std = X.std(dim=0)
        self.transform = transform

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        x = (self.X[idx] - self.mean) / (self.std + 1e-8)
        if self.transform is not None:
            x = self.transform(x)
        return x, self.y[idx]


def add_noise(x):
    return x + 0.01 * torch.randn_like(x)


norm_dataset = NormalizedDataset(custom_X, custom_y, transform=add_noise)
nx, ny = norm_dataset[0]
assert nx.shape == (8,) and ny.ndim == 0
print("transformed sample shape:", nx.shape)

## 13. Train/Validation Split and Loaders 🟢 Easy

### 🔥 Interview Essential: train vs validation loaders

A deterministic tensor slice is simple, while `random_split` plus a seeded generator creates a reproducible random partition.

In [ ]:
split_X = torch.randn(1000, 10)
split_y = torch.randint(0, 3, (1000,))
split_at = int(0.8 * len(split_X))
X_train, y_train = split_X[:split_at], split_y[:split_at]
X_val, y_val = split_X[split_at:], split_y[split_at:]
assert len(X_train) == 800 and len(X_val) == 200

full_dataset = TensorDataset(split_X, split_y)
train_dataset, val_dataset = random_split(
    full_dataset, [800, 200],
    generator=torch.Generator().manual_seed(42),
)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=0)
assert len(train_dataset) == 800 and len(val_dataset) == 200
print("train batches:", len(train_loader), "validation batches:", len(val_loader))

In [ ]:
# Exercise 3 — split 60 samples reproducibly into 45 train and 15 validation.
small_full = TensorDataset(torch.randn(60, 4), torch.randint(0, 2, (60,)))
small_train, small_val = random_split(
    small_full, [45, 15], generator=torch.Generator().manual_seed(7)
)
assert len(small_train) == 45 and len(small_val) == 15
print("✅ random_split exercise correct")

## 14. DataLoader + Training Loop 🟢 Easy

DataLoader's role is to provide batches. The training loop still clears gradients, runs the model, computes loss, backpropagates, and updates parameters.

In [ ]:
demo_model = nn.Linear(10, 3)
demo_loss_fn = nn.CrossEntropyLoss()
demo_optimizer = torch.optim.SGD(demo_model.parameters(), lr=0.05)

demo_model.train()
X_batch, y_batch = next(iter(train_loader))
demo_optimizer.zero_grad()
logits = demo_model(X_batch)
loss = demo_loss_fn(logits, y_batch)
loss.backward()
demo_optimizer.step()
print("one training step loss:", round(loss.item(), 4))

## 15. Custom Batch Structures 🟡 Medium

A Dataset need not return only `(X, y)`. It can return a dictionary, and default collation recursively batches compatible fields. This is common in modern NLP/LLM code.

In [ ]:
class DictionaryDataset(Dataset):
    def __init__(self, X, y):
        self.X, self.y = X, y

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return {"features": self.X[idx], "label": self.y[idx]}


dict_loader = DataLoader(DictionaryDataset(custom_X, custom_y), batch_size=7)
dict_batch = next(iter(dict_loader))
print({key: value.shape for key, value in dict_batch.items()})
assert dict_batch["features"].shape == (7, 8)
assert dict_batch["label"].shape == (7,)

## 16. `num_workers`, `persistent_workers`, and Prefetching 🟡 Medium

### 🔥 Interview Essential: `num_workers`

- `num_workers=0`: the main process loads samples.
- `num_workers>0`: worker processes load/preprocess samples in parallel.

Workers can overlap data preparation with training and keep a GPU fed, but more is not automatically faster: process overhead, memory, CPU contention, storage contention, batch size, and platform behavior matter. Benchmark realistic values.

`persistent_workers=True` keeps workers alive between epochs, which can reduce startup cost; it requires `num_workers>0`. `prefetch_factor` controls how many batches each worker prepares ahead. Both are optional/intermediate tuning tools. Runnable exercises keep workers at zero for portability.

In [ ]:
portable_loader = DataLoader(custom_dataset, batch_size=8, num_workers=0)
assert portable_loader.num_workers == 0
print("portable loader batches:", len(portable_loader))

## 17. `pin_memory` and Device Handling 🟡 Medium

### 🔥 Interview Essential: device-aware data flow

Pinned CPU memory can speed CPU→CUDA transfer. A typical CUDA loader uses `pin_memory=True`, followed by `batch.to("cuda", non_blocking=True)`. Pinning does **not** move a tensor to GPU and is not universally beneficial for CPU-only training.

DataLoader usually yields CPU tensors. Move only the current batch inside the loop; move the model once before the loop. Loading an entire dataset onto GPU by default can exhaust memory, conflicts with the usual CPU worker pipeline, and prevents streaming.

```text
disk / CPU Dataset → DataLoader → CPU batch → .to(device) → model
```

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device_loader = DataLoader(custom_dataset, batch_size=8, pin_memory=torch.cuda.is_available())
device_model = nn.Linear(8, 4).to(device)

batch_X, batch_y = next(iter(device_loader))
batch_X = batch_X.to(device, non_blocking=torch.cuda.is_available())
batch_y = batch_y.to(device, non_blocking=torch.cuda.is_available())
assert batch_X.device == next(device_model.parameters()).device
print("batch/model device:", device)

## 18. `collate_fn` and Variable-Length Data 🟡 Medium

### 🔥 Interview Essential, especially for NLP

`collate_fn` defines how a list of individual samples becomes one batch. Default collation stacks same-shaped tensors. Variable-length sequences cannot be stacked directly, so a custom collator can dynamically pad only to the longest sequence in the current batch.

In [ ]:
sequences = [
    torch.tensor([1, 2, 3]),
    torch.tensor([4, 5]),
    torch.tensor([6, 7, 8, 9]),
]


class SequenceDataset(Dataset):
    def __init__(self, sequences):
        self.sequences = sequences

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):
        return self.sequences[idx]


sequence_dataset = SequenceDataset(sequences)
try:
    next(iter(DataLoader(sequence_dataset, batch_size=3)))
except RuntimeError as error:
    print("Expected default-collation failure:", str(error).splitlines()[0])

In [ ]:
def collate_sequences(batch):
    max_len = max(len(sequence) for sequence in batch)
    padded = torch.zeros(len(batch), max_len, dtype=torch.long)
    for i, sequence in enumerate(batch):
        padded[i, :len(sequence)] = sequence
    return padded


sequence_loader = DataLoader(
    sequence_dataset, batch_size=3, collate_fn=collate_sequences
)
padded = next(iter(sequence_loader))
expected = torch.tensor([[1, 2, 3, 0], [4, 5, 0, 0], [6, 7, 8, 9]])
print(padded)
assert torch.equal(padded, expected)

## 19. `pad_sequence`, Attention Masks, and NLP Batches 🟡 Medium

`pad_sequence(..., batch_first=True)` is a convenient dynamic-padding helper. If token ID 0 is reserved for padding, `tokens != 0` creates a Boolean attention mask with the same `(batch, sequence_length)` shape. It marks real tokens for pooling/attention; this notebook does not teach transformer attention itself.

In [ ]:
nlp_items = [
    {"tokens": torch.tensor([8, 2, 5]), "label": 1},
    {"tokens": torch.tensor([7, 4]), "label": 0},
    {"tokens": torch.tensor([3, 9, 6, 2]), "label": 2},
]


def collate_nlp(batch):
    tokens = pad_sequence(
        [item["tokens"] for item in batch],
        batch_first=True,
        padding_value=0,
    )
    attention_mask = tokens != 0
    labels = torch.tensor([item["label"] for item in batch], dtype=torch.long)
    return {"input_ids": tokens, "attention_mask": attention_mask, "labels": labels}


nlp_loader = DataLoader(nlp_items, batch_size=3, collate_fn=collate_nlp)
nlp_batch = next(iter(nlp_loader))
print(nlp_batch)
assert nlp_batch["input_ids"].shape == (3, 4)
assert nlp_batch["attention_mask"].shape == (3, 4)
assert nlp_batch["labels"].shape == (3,)

## 20. Dataset Patterns: Tabular, Images, and Text

### Tabular 🟢 Easy

Features commonly use `float32`; multiclass labels commonly use `long`/`int64` for `CrossEntropyLoss`.

In [ ]:
class TabularDataset(Dataset):
    def __init__(self, X, y):
        self.X = X.float()
        self.y = y.long()

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


tabular_dataset = TabularDataset(torch.randn(12, 3, dtype=torch.float64), torch.arange(12) % 2)
tx, ty = tabular_dataset[0]
assert tx.dtype == torch.float32 and ty.dtype == torch.int64

### Images 🟡 Medium (conceptual; no external files)

```python
class ImageDataset(Dataset):
    def __init__(self, image_paths, labels, transform=None):
        self.image_paths = image_paths
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        image = load_image(self.image_paths[idx])  # project-specific loader
        if self.transform:
            image = self.transform(image)
        return image, self.labels[idx]
```

Store filenames → load one image → transform → tensor → label. For a million disk images, usually store paths rather than loading all images into RAM in `__init__`.

### Text 🟡 Medium

Raw text → tokenization → token IDs → dynamic padding → attention mask → model. Token sequences can be stored individually and custom `collate_fn` can batch them.

In [ ]:
class TextDataset(Dataset):
    def __init__(self, token_sequences, labels):
        self.tokens, self.labels = token_sequences, labels

    def __len__(self):
        return len(self.tokens)

    def __getitem__(self, idx):
        return {"tokens": self.tokens[idx], "label": self.labels[idx]}


text_dataset = TextDataset(sequences, [0, 1, 0])
assert set(text_dataset[0]) == {"tokens", "label"}

## 21. Map-Style vs Iterable Datasets and Samplers 🔴 Hard / Optional

The custom Datasets above are **map-style**: `dataset[index]`. `IterableDataset` supports streaming, sequential files, generated-on-the-fly data, or data too large for random access.

A `Sampler` determines which indices are retrieved and in what order. `shuffle=True` is enough for many cases. A custom sampler enables specialized strategies; `WeightedRandomSampler` makes high-weight samples more likely. This changes sampling frequency and is different from a class-weighted loss. Do not normally specify both `shuffle=True` and a sampler controlling order—the arguments are incompatible.

In [ ]:
# Optional balanced-sampling demonstration on an imbalanced label vector.
imbalanced_y = torch.tensor([0] * 18 + [1] * 2)
imbalanced_X = torch.arange(20).float().unsqueeze(1)
class_counts = torch.bincount(imbalanced_y)
class_weights = 1.0 / class_counts.float()
sample_weights = class_weights[imbalanced_y]
sampler = WeightedRandomSampler(
    sample_weights, num_samples=len(sample_weights), replacement=True,
    generator=torch.Generator().manual_seed(42),
)
balanced_loader = DataLoader(
    TensorDataset(imbalanced_X, imbalanced_y), batch_size=10, sampler=sampler
)
sampled_labels = torch.cat([yb for _, yb in balanced_loader])
print("original counts:", class_counts.tolist())
print("one weighted epoch counts:", torch.bincount(sampled_labels, minlength=2).tolist())

## 22. Performance Pipeline Mental Model 🟡 Medium

```text
Storage
   ↓
Dataset.__getitem__
   ↓
DataLoader workers
   ↓
collate_fn
   ↓
CPU batch
   ↓
device transfer
   ↓
Model
```

If GPU utilization is 30% and the GPU waits between batches, training may be data-loader-bound. Potential experiments include benchmarking workers, cheaper preprocessing, caching, faster storage, pinned memory, or prefetching. None is an automatic fix; measure end-to-end throughput.

## 23. Debugging Exercises (20) 🟡 Medium

**1. Diagnose:** `__len__` returns `self.X.shape[1]`.

<details><summary>Fix</summary>Return the sample count, normally `self.X.shape[0]` or `len(self.X)`.</details>

**2. Diagnose:** `__getitem__` returns `self.X, self.y`.

<details><summary>Fix</summary>Index both objects and return one sample: `self.X[idx], self.y[idx]`.</details>

**3. Diagnose:** `X` has 100 rows but `y` has 90 labels.

<details><summary>Fix</summary>Lengths are misaligned; fix the data before constructing the Dataset.</details>

**4. Diagnose:** A standard training loader uses `shuffle=False` accidentally.

<details><summary>Fix</summary>Usually change it to `shuffle=True` so order varies by epoch.</details>

**5. Diagnose:** A validation loader applies stochastic training augmentation.

<details><summary>Fix</summary>Disable or replace stochastic training augmentation for stable evaluation.</details>

**6. Diagnose:** Code assumes every batch contains exactly 64 samples.

<details><summary>Fix</summary>The final batch may be smaller; derive sizes from the current batch or use justified `drop_last=True`.</details>

**7. Diagnose:** Validation uses `drop_last=True`.

<details><summary>Fix</summary>Some validation samples are omitted; normally use `drop_last=False`.</details>

**8. Diagnose:** Samples return inconsistent tensor shapes.

<details><summary>Fix</summary>Default collation cannot stack them; normalize shapes or provide a custom collator.</details>

**9. Diagnose:** Variable-length sequences use default collation.

<details><summary>Fix</summary>Pad dynamically in `collate_fn`, for example with `pad_sequence`.</details>

**10. Diagnose:** A tokenizer/model is initialized inside every `__getitem__` call.

<details><summary>Fix</summary>Move reusable initialization out of the per-sample path.</details>

**11. Diagnose:** `num_workers=64` makes loading slower and memory usage spikes.

<details><summary>Fix</summary>Too many workers create contention/overhead; benchmark smaller values.</details>

**12. Diagnose:** `pin_memory=True` is expected to put a batch on CUDA.

<details><summary>Fix</summary>Pinning only changes CPU memory allocation; still call `.to(device)`.</details>

**13. Diagnose:** CUDA model receives CPU inputs.

<details><summary>Fix</summary>Move each input/target batch to the model device before forward.</details>

**14. Diagnose:** `model.to(device)` is called inside every batch iteration.

<details><summary>Fix</summary>Move the model once before entering the loop.</details>

**15. Diagnose:** Multiclass targets for `CrossEntropyLoss` are `float32`.

<details><summary>Fix</summary>Use class indices with `torch.long` dtype.</details>

**16. Diagnose:** A loader sets both `shuffle=True` and a custom sampler.

<details><summary>Fix</summary>Remove `shuffle`; let the sampler control index selection/order.</details>

**17. Diagnose:** Every sentence is globally padded to 50,000 tokens.

<details><summary>Fix</summary>Use dynamic per-batch padding to reduce wasted memory and compute.</details>

**18. Diagnose:** Dataset-wide mean and standard deviation are recomputed per sample.

<details><summary>Fix</summary>Compute reusable statistics once, not inside every `__getitem__`.</details>

**19. Diagnose:** Labels are returned in a Python object structure that varies per sample.

<details><summary>Fix</summary>Return a consistent structure/type so collation can batch it.</details>

**20. Diagnose:** A shuffled validation loader is assumed to change validation accuracy.

<details><summary>Fix</summary>Order usually does not change aggregate metrics; no shuffle is preferred for reproducibility/alignment.</details>

In [ ]:
# A compact executable repair of the two classic custom-Dataset bugs.
class RepairedDataset(Dataset):
    def __init__(self, X, y):
        assert len(X) == len(y)
        self.X, self.y = X, y

    def __len__(self):
        return self.X.shape[0]

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


repaired = RepairedDataset(torch.randn(9, 4), torch.arange(9))
assert len(repaired) == 9 and repaired[0][0].shape == (4,)

## 24. Interview Questions (35)

### 1. What is a PyTorch Dataset?

<details><summary>Short Interview Answer</summary>

An object representing samples and defining how one sample is retrieved.
</details>

<details><summary>Detailed Explanation</summary>

A map-style Dataset normally implements `__len__` and `__getitem__`; it separates sample access from batching.
</details>

### 2. What is a DataLoader?

<details><summary>Short Interview Answer</summary>

An iterable that batches and serves samples from a Dataset.
</details>

<details><summary>Detailed Explanation</summary>

It coordinates index sampling, optional shuffling/workers, and collation into batches.
</details>

### 3. Dataset vs DataLoader?

<details><summary>Short Interview Answer</summary>

Dataset defines one-sample access; DataLoader handles batched iteration.
</details>

<details><summary>Detailed Explanation</summary>

Keep storage/retrieval logic in Dataset and batching/order/loading mechanics in DataLoader.
</details>

### 4. What methods must a standard custom Dataset implement?

<details><summary>Short Interview Answer</summary>

Usually `__len__` and `__getitem__`.
</details>

<details><summary>Detailed Explanation</summary>

It commonly also has `__init__` to store paths, tensors, labels, and transforms.
</details>

### 5. What does `__len__()` do?

<details><summary>Short Interview Answer</summary>

Returns the number of samples.
</details>

<details><summary>Detailed Explanation</summary>

`len(dataset)` calls it; loaders and samplers use it for map-style datasets.
</details>

### 6. What does `__getitem__()` do?

<details><summary>Short Interview Answer</summary>

Returns the sample at an index.
</details>

<details><summary>Detailed Explanation</summary>

DataLoader repeatedly invokes it for sampled indices, possibly in worker processes.
</details>

### 7. What is TensorDataset?

<details><summary>Short Interview Answer</summary>

A Dataset that indexes aligned tensors together.
</details>

<details><summary>Detailed Explanation</summary>

It is a concise choice when data already lives in tensors with equal leading length.
</details>

### 8. When use a custom Dataset instead of TensorDataset?

<details><summary>Short Interview Answer</summary>

When retrieval, files, transforms, metadata, or output structures need custom logic.
</details>

<details><summary>Detailed Explanation</summary>

TensorDataset only indexes preexisting aligned tensors; custom Dataset can load and preprocess on demand.
</details>

### 9. What is batch size?

<details><summary>Short Interview Answer</summary>

The number of samples processed in one training iteration.
</details>

<details><summary>Detailed Explanation</summary>

It affects memory, gradient noise, optimizer steps per epoch, and hardware efficiency.
</details>

### 10. Why use mini-batches?

<details><summary>Short Interview Answer</summary>

They balance vectorized efficiency, memory use, and useful stochastic gradients.
</details>

<details><summary>Detailed Explanation</summary>

They avoid full-dataset memory cost while using hardware better than single-sample updates.
</details>

### 11. What does `shuffle=True` do?

<details><summary>Short Interview Answer</summary>

Randomizes sample index order for an iteration/epoch.
</details>

<details><summary>Detailed Explanation</summary>

It changes order, not sample values, and is not augmentation.
</details>

### 12. Why shuffle training data?

<details><summary>Short Interview Answer</summary>

To avoid repeatedly training in a fixed sample order.
</details>

<details><summary>Detailed Explanation</summary>

It improves mixing and makes mini-batch gradients less tied to data ordering.
</details>

### 13. Must validation data be shuffled?

<details><summary>Short Interview Answer</summary>

Usually no.
</details>

<details><summary>Detailed Explanation</summary>

No parameter updates occur; stable order supports reproducibility and prediction-to-sample alignment.
</details>

### 14. What happens to the final incomplete batch?

<details><summary>Short Interview Answer</summary>

It is kept by default.
</details>

<details><summary>Detailed Explanation</summary>

With 10 samples and batch size 4, sizes are 4, 4, and 2.
</details>

### 15. What does `drop_last=True` do?

<details><summary>Short Interview Answer</summary>

Drops the final incomplete batch.
</details>

<details><summary>Detailed Explanation</summary>

Only full-size batches remain, and discarded samples are absent for that epoch.
</details>

### 16. Why might `drop_last=True` be used?

<details><summary>Short Interview Answer</summary>

When fixed batch shapes or avoiding a tiny final training batch matters.
</details>

<details><summary>Detailed Explanation</summary>

Some BatchNorm, compilation, or distributed setups may motivate it; use deliberately.
</details>

### 17. Why avoid `drop_last=True` for validation?

<details><summary>Short Interview Answer</summary>

It omits samples from metrics.
</details>

<details><summary>Detailed Explanation</summary>

Evaluation normally should cover every validation/test example.
</details>

### 18. What does `num_workers` control?

<details><summary>Short Interview Answer</summary>

The number of worker processes used for sample loading/preprocessing.
</details>

<details><summary>Detailed Explanation</summary>

Zero loads in the main process; positive values enable parallel workers.
</details>

### 19. Does larger `num_workers` always improve speed?

<details><summary>Short Interview Answer</summary>

No; benchmark it.
</details>

<details><summary>Detailed Explanation</summary>

Overhead, CPU/memory/storage contention, preprocessing, and platform determine the best value.
</details>

### 20. What does `pin_memory=True` do?

<details><summary>Short Interview Answer</summary>

Allocates returned CPU tensors in pinned memory where supported.
</details>

<details><summary>Detailed Explanation</summary>

This may improve CPU-to-CUDA transfers, especially with non-blocking copies.
</details>

### 21. When is pinned memory useful?

<details><summary>Short Interview Answer</summary>

Primarily for CPU batches transferred to CUDA.
</details>

<details><summary>Detailed Explanation</summary>

It is not automatically useful for CPU-only training and should be measured.
</details>

### 22. Does pinning move tensors to GPU?

<details><summary>Short Interview Answer</summary>

No.
</details>

<details><summary>Detailed Explanation</summary>

The training loop must still call `.to(device)`.
</details>

### 23. What does `collate_fn` do?

<details><summary>Short Interview Answer</summary>

Combines a list of samples into a batch.
</details>

<details><summary>Detailed Explanation</summary>

Default collation stacks compatible tensors; custom logic can pad or construct dictionaries.
</details>

### 24. Why is custom collation useful for NLP?

<details><summary>Short Interview Answer</summary>

Sequences often have different lengths.
</details>

<details><summary>Detailed Explanation</summary>

A collator can dynamically pad, create masks, and stack labels per batch.
</details>

### 25. How do you batch variable-length sequences?

<details><summary>Short Interview Answer</summary>

Pad them in a custom `collate_fn`.
</details>

<details><summary>Detailed Explanation</summary>

Use the current batch maximum length and optionally return original lengths or masks.
</details>

### 26. What is dynamic padding?

<details><summary>Short Interview Answer</summary>

Padding only to the longest sequence in the current batch.
</details>

<details><summary>Detailed Explanation</summary>

It often wastes less compute than padding every sample to one global maximum.
</details>

### 27. What does `pad_sequence` do?

<details><summary>Short Interview Answer</summary>

Pads a list of variable-length tensors to a common length.
</details>

<details><summary>Detailed Explanation</summary>

`batch_first=True` returns a batch-leading layout.
</details>

### 28. Map-style Dataset vs IterableDataset?

<details><summary>Short Interview Answer</summary>

Map-style supports indexed access; iterable style yields a stream.
</details>

<details><summary>Detailed Explanation</summary>

Iterable datasets suit streams or naturally sequential/very large sources.
</details>

### 29. What is a sampler?

<details><summary>Short Interview Answer</summary>

It determines which dataset indices are retrieved and in what order.
</details>

<details><summary>Detailed Explanation</summary>

Weighted or distributed strategies can go beyond ordinary shuffling.
</details>

### 30. Why not move the whole Dataset to GPU by default?

<details><summary>Short Interview Answer</summary>

It may not fit and bypasses the usual CPU loading pipeline.
</details>

<details><summary>Detailed Explanation</summary>

Move only current batches so data can stream and CPU workers can prepare it.
</details>

### 31. Where should `.to(device)` usually happen?

<details><summary>Short Interview Answer</summary>

Inside the loop on each batch, before model forward.
</details>

<details><summary>Detailed Explanation</summary>

Move the model once before the loop and each batch as it arrives.
</details>

### 32. What can make DataLoader the bottleneck?

<details><summary>Short Interview Answer</summary>

Slow I/O, preprocessing, collation, storage, or insufficient loading concurrency.
</details>

<details><summary>Detailed Explanation</summary>

The accelerator may wait idle between batches even when model compute is fast.
</details>

### 33. How might you speed up a slow input pipeline?

<details><summary>Short Interview Answer</summary>

Profile, then try workers, faster preprocessing/storage, caching, pinning, or prefetching.
</details>

<details><summary>Detailed Explanation</summary>

Changes depend on the measured bottleneck; more workers are not universally better.
</details>

### 34. What if Dataset samples have inconsistent shapes?

<details><summary>Short Interview Answer</summary>

Default tensor stacking usually fails.
</details>

<details><summary>Detailed Explanation</summary>

Normalize shapes or define custom collation such as dynamic padding.
</details>

### 35. What dtype should multiclass labels commonly use?

<details><summary>Short Interview Answer</summary>

`torch.long` (`int64`).
</details>

<details><summary>Detailed Explanation</summary>

`CrossEntropyLoss` expects integer class indices in its standard form.
</details>

## 25. Knowledge Check Quiz (28)

**1. Which method reports Dataset size?**

A. `__init__`
B. `__getitem__`
C. `__len__`
D. `forward`

**2. What does DataLoader normally do?**

A. Defines architecture
B. Computes gradients
C. Batches/iterates samples
D. Updates parameters

**3. 100 samples, batch 32, `drop_last=False`: batches?**

A. 3
B. 4
C. 32
D. 100

**4. 10 samples, batch 4, `drop_last=True`: samples observed?**

A. 8
B. 10
C. 4
D. 6

**5. Typical training shuffle setting?**

A. True
B. False
C. None
D. Only on validation

**6. Typical validation shuffle setting?**

A. Always true
B. Usually false
C. Must be random
D. Not supported

**7. `TensorDataset` requires aligned tensors to share which size?**

A. Last dimension
B. All dimensions
C. First dimension
D. Dtype

**8. What should `dataset[i]` normally return?**

A. Entire epoch
B. One sample
C. An optimizer
D. A gradient

**9. `num_workers=0` means?**

A. No data
B. Main-process loading
C. GPU loading
D. Zero batches

**10. Does more workers always help?**

A. Yes
B. Only on CPU
C. No
D. Only with shuffle

**11. Pinning memory automatically moves data to CUDA?**

A. Yes
B. No
C. Only labels
D. Only models

**12. Where are batches normally moved to device?**

A. Inside Dataset constructor
B. Inside training loop
C. Inside loss
D. Never

**13. Default collation works best when tensor samples are?**

A. Same shape
B. Different shape
C. Strings only
D. On different devices

**14. Variable-length sequences usually need?**

A. A larger loss
B. Dynamic padding/custom collation
C. More classes
D. No Dataset

**15. `pad_sequence(..., batch_first=True)` places batch dimension?**

A. First
B. Last
C. Removes it
D. Randomly

**16. An attention mask from padded IDs can be?**

A. `ids == 0` for real tokens
B. `ids != 0` for real tokens
C. `ids.float()` only
D. A scalar

**17. Shuffling is the same as augmentation?**

A. Yes
B. No
C. Only for images
D. Only on GPU

**18. A sampler controls?**

A. Model weights
B. Index selection/order
C. Loss dtype
D. Gradient clipping

**19. Can `shuffle=True` normally be combined with a custom sampler?**

A. Yes, always
B. No, they conflict
C. Only for text
D. Only with zero workers

**20. For multiclass CrossEntropy targets, common dtype?**

A. float64
B. bool
C. long
D. complex

**21. A map-style Dataset supports?**

A. `dataset[index]`
B. Only streaming
C. Only images
D. Optimizer steps

**22. IterableDataset is suited to?**

A. Only tiny tensors
B. Streams/sequential sources
C. Loss functions
D. Model layers

**23. Persistent workers primarily avoid?**

A. Batching
B. Worker restart between epochs
C. Labels
D. Shuffling

**24. `prefetch_factor` concerns?**

A. Batches prepared ahead
B. Model depth
C. Loss scale
D. Learning rate

**25. Why avoid global maximum padding?**

A. It changes labels
B. It may waste compute/memory
C. It disables gradients
D. It removes batches

**26. Why avoid expensive global stats in `__getitem__`?**

A. It runs repeatedly
B. It changes dtype only
C. It disables shuffle
D. It forces CUDA

**27. A 7-sample dataset with batch size 4 yields final batch size?**

A. 4
B. 3
C. 7
D. 0

**28. WeightedRandomSampler differs from class-weighted loss because it changes?**

A. Sampling frequency
B. Model shape
C. Target dtype
D. Device

### Quiz Answers

1. **C** · 2. **C** · 3. **B** · 4. **A** · 5. **A** · 6. **B** · 7. **C** · 8. **B** · 9. **B** · 10. **C** · 11. **B** · 12. **B** · 13. **A** · 14. **B** · 15. **A** · 16. **B** · 17. **B** · 18. **B** · 19. **B** · 20. **C** · 21. **A** · 22. **B** · 23. **B** · 24. **A** · 25. **B** · 26. **A** · 27. **B** · 28. **A**

## 26. Write From Memory (20 prompts)

1. Import `Dataset` and `DataLoader`.
2. Create `TensorDataset(X, y)`.
3. Create a training loader with batch size 32 and shuffling.
4. Create a validation loader with batch size 64 and no shuffling.
5. Write a Dataset with `__init__`, `__len__`, and `__getitem__`.
6. Return one feature-label pair from `__getitem__`.
7. Split a Dataset reproducibly with `random_split`.
8. Create a loader with `drop_last=True`.
9. Create a loader configured with four workers (conceptually).
10. Create a loader using pinned memory.
11. Iterate and move feature/target batches to `device`.
12. Write a collator that pads sequences.
13. Generate an attention mask for padding ID zero.
14. Return a batched dictionary with IDs, mask, and labels.
15. Compute the number of batches with and without dropping.
16. Inspect only the first DataLoader batch.
17. Write a Dataset accepting an optional transform.
18. Create a dictionary-returning Dataset.
19. Create a weighted sampler from per-sample weights.
20. Explain the complete storage-to-model data path aloud.

In [ ]:
# Reference implementations for several practical memory prompts.
memory_dataset_2 = TensorDataset(torch.randn(20, 5), torch.randint(0, 2, (20,)))
memory_train_loader = DataLoader(memory_dataset_2, batch_size=4, shuffle=True)
memory_val_loader = DataLoader(memory_dataset_2, batch_size=8, shuffle=False)
memory_drop_loader = DataLoader(memory_dataset_2, batch_size=6, drop_last=True)

assert next(iter(memory_train_loader))[0].shape == (4, 5)
assert next(iter(memory_val_loader))[0].shape == (8, 5)
assert len(memory_drop_loader) == 3
print("✅ Core memory prompts validated")

### Write a Custom Dataset From Memory — attempt 2

Repeat without scrolling to section 11. Include an optional transform this time.

In [ ]:
class MyDatasetAgain(Dataset):
    def __init__(self, X, y, transform=None):
        self.X, self.y, self.transform = X, y, transform

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        x = self.X[idx]
        if self.transform is not None:
            x = self.transform(x)
        return x, self.y[idx]


again = MyDatasetAgain(torch.ones(11, 2), torch.arange(11), transform=lambda x: x * 3)
assert len(again) == 11
assert torch.equal(again[4][0], torch.tensor([3.0, 3.0]))
assert again[4][1].item() == 4
print("✅ Second memory implementation validated")

## 27. Data Pipeline Reading Exercise

Explain every argument verbally:

```python
train_loader = DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True,
    num_workers=4,
    pin_memory=True,
)
```

<details><summary>Expected explanation</summary>

- `train_dataset`: source of samples
- `64`: 64 samples per batch
- `shuffle=True`: randomize training index order
- `num_workers=4`: four worker processes load samples
- `pin_memory=True`: pinned CPU memory may speed CUDA transfer

</details>

## 28. Data Pipeline Design Scenarios (10)

**Scenario 1:** A small tabular dataset already stored as tensors. Custom Dataset or TensorDataset?

<details><summary>Likely answer</summary>`TensorDataset` is sufficient unless custom behavior is needed.</details>

**Scenario 2:** One million image filenames live on disk. Load every image in `__init__`?

<details><summary>Likely answer</summary>Usually no—store paths and load samples on demand.</details>

**Scenario 3:** Sentences have different lengths. What loader feature helps?

<details><summary>Likely answer</summary>A custom `collate_fn` with dynamic padding.</details>

**Scenario 4:** GPU waits between batches. What should you investigate?

<details><summary>Likely answer</summary>Profile I/O/preprocessing; benchmark workers, caching, storage, pinning, and prefetching.</details>

**Scenario 5:** Validation predictions must align with an external ID list.

<details><summary>Likely answer</summary>Keep deterministic order, normally `shuffle=False`, and return IDs if useful.</details>

**Scenario 6:** The last training batch has one item and BatchNorm is unstable.

<details><summary>Likely answer</summary>Consider `drop_last=True` for training after confirming sample loss is acceptable.</details>

**Scenario 7:** Evaluation currently drops its final 13 examples.

<details><summary>Likely answer</summary>Use `drop_last=False` so metrics cover all samples.</details>

**Scenario 8:** A transform creates a different random crop on validation every run.

<details><summary>Likely answer</summary>Use deterministic validation preprocessing rather than stochastic training augmentation.</details>

**Scenario 9:** Class 1 has far fewer examples than class 0.

<details><summary>Likely answer</summary>Consider weighted sampling or a class-weighted loss; they solve related but different problems.</details>

**Scenario 10:** Adding workers slows a tiny in-memory tensor dataset.

<details><summary>Likely answer</summary>Worker overhead can dominate; `num_workers=0` may be best.</details>

## 29. Final Data Pipeline Challenge 🔴 Hard

Build a complete synthetic text-classification pipeline: variable-length token sequences → custom Dataset → dynamic collator → train/validation loaders → embedding → masked mean pooling → classifier → training.

In [ ]:
torch.manual_seed(42)
num_samples = 200
vocab_size = 1000
num_classes = 3

token_sequences = []
token_labels = []
for _ in range(num_samples):
    length = int(torch.randint(5, 26, (1,)).item())
    # Zero is reserved for padding, so real IDs start at 1.
    token_sequences.append(torch.randint(1, vocab_size, (length,), dtype=torch.long))
    token_labels.append(int(torch.randint(0, num_classes, (1,)).item()))

assert len(token_sequences) == 200
assert all(5 <= len(seq) <= 25 and seq.dtype == torch.long for seq in token_sequences)

In [ ]:
class TokenDataset(Dataset):
    def __init__(self, sequences, labels):
        if len(sequences) != len(labels):
            raise ValueError("sequences and labels must be aligned")
        self.sequences = sequences
        self.labels = labels

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):
        return {"tokens": self.sequences[idx], "label": self.labels[idx]}


token_dataset = TokenDataset(token_sequences, token_labels)
for idx in torch.randint(0, num_samples, (5,)).tolist():
    item = token_dataset[idx]
    assert set(item) == {"tokens", "label"}
    assert item["tokens"].ndim == 1 and item["tokens"].dtype == torch.long
    assert isinstance(item["label"], int)
print("✅ TokenDataset validated")

In [ ]:
def collate_batch(batch):
    input_ids = pad_sequence(
        [item["tokens"] for item in batch],
        batch_first=True,
        padding_value=0,
    )
    attention_mask = input_ids != 0
    labels = torch.tensor([item["label"] for item in batch], dtype=torch.long)
    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels,
    }


challenge_train, challenge_val = random_split(
    token_dataset, [160, 40], generator=torch.Generator().manual_seed(42)
)
challenge_train_loader = DataLoader(
    challenge_train, batch_size=16, shuffle=True,
    collate_fn=collate_batch, num_workers=0,
)
challenge_val_loader = DataLoader(
    challenge_val, batch_size=32, shuffle=False,
    collate_fn=collate_batch, num_workers=0,
)

batch = next(iter(challenge_train_loader))
assert batch["input_ids"].ndim == 2
assert batch["attention_mask"].shape == batch["input_ids"].shape
assert batch["labels"].ndim == 1
assert batch["input_ids"].dtype == torch.long
assert torch.equal(batch["attention_mask"], batch["input_ids"] != 0)
assert torch.all(batch["input_ids"][~batch["attention_mask"]] == 0)
print({key: value.shape for key, value in batch.items()})

In [ ]:
class TextClassifier(nn.Module):
    def __init__(self, vocab_size, embedding_dim, num_classes):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        self.classifier = nn.Linear(embedding_dim, num_classes)

    def forward(self, input_ids, attention_mask):
        embedded = self.embedding(input_ids)               # (batch, sequence, embedding)
        mask = attention_mask.unsqueeze(-1).to(embedded.dtype)
        summed = (embedded * mask).sum(dim=1)
        lengths = mask.sum(dim=1).clamp_min(1.0)
        pooled = summed / lengths                          # masked mean pooling
        return self.classifier(pooled)


text_model = TextClassifier(vocab_size, embedding_dim=32, num_classes=num_classes)
test_logits = text_model(batch["input_ids"], batch["attention_mask"])
assert test_logits.shape == (batch["labels"].shape[0], num_classes)
print("model output:", test_logits.shape)

In [ ]:
challenge_loss_fn = nn.CrossEntropyLoss()
challenge_optimizer = torch.optim.AdamW(text_model.parameters(), lr=0.01)

for epoch in range(3):
    text_model.train()
    running_loss = 0.0
    for batch in challenge_train_loader:
        input_ids = batch["input_ids"]
        attention_mask = batch["attention_mask"]
        labels = batch["labels"]

        challenge_optimizer.zero_grad()
        logits = text_model(input_ids, attention_mask)
        loss = challenge_loss_fn(logits, labels)
        loss.backward()
        challenge_optimizer.step()
        running_loss += loss.item() * labels.shape[0]

    text_model.eval()
    correct = total = 0
    with torch.no_grad():
        for batch in challenge_val_loader:
            logits = text_model(batch["input_ids"], batch["attention_mask"])
            correct += (logits.argmax(dim=1) == batch["labels"]).sum().item()
            total += batch["labels"].numel()

    print(
        f"epoch {epoch + 1}: train_loss={running_loss / len(challenge_train):.4f}, "
        f"val_accuracy={correct / total:.3f}"
    )

assert total == 40
print("✅ Full variable-length data pipeline executed")

## 30. Cheat Sheet

```python
from torch.utils.data import Dataset, TensorDataset, DataLoader, random_split

dataset = TensorDataset(X, y)

class MyDataset(Dataset):
    def __init__(self, X, y):
        self.X, self.y = X, y
    def __len__(self):
        return len(self.X)
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

# drop_last=True       # discard final partial batch only with a reason
# num_workers=4        # parallel CPU loading; benchmark
# pin_memory=True      # may accelerate CPU→CUDA transfer

for X, y in train_loader:
    X, y = X.to(device), y.to(device)
```

Variable-length batching:

```python
def collate_fn(batch):
    sequences = [item["tokens"] for item in batch]
    labels = torch.tensor([item["label"] for item in batch])
    padded = pad_sequence(sequences, batch_first=True, padding_value=0)
    return {
        "input_ids": padded,
        "attention_mask": padded != 0,
        "labels": labels,
    }
```

## 31. Critical Mental Model and Interview Quick Reference

```text
Storage / tensors
       ↓
Dataset → __getitem__ → one sample
       ↓
collate_fn
       ↓
DataLoader → batch
       ↓
.to(device)
       ↓
model
```

| Term | Immediate meaning |
|---|---|
| Dataset | how to retrieve one sample |
| `__len__` | number of samples |
| `__getitem__` | retrieve sample by index |
| DataLoader | batching + iteration |
| `batch_size` | samples per batch |
| `shuffle` | randomize index order |
| `drop_last` | discard final partial batch |
| `num_workers` | parallel CPU data loading |
| `pin_memory` | potentially faster CPU→CUDA transfer |
| `collate_fn` | combine samples into a batch |
| TensorDataset | simple Dataset for aligned tensors |

**Training:** `shuffle=True`  
**Validation:** `shuffle=False`

## 32. Before Moving to 08_gpu_and_devices.ipynb

- [ ] I can explain Dataset vs DataLoader.
- [ ] I can create a TensorDataset and DataLoader.
- [ ] I understand batch size, shuffling, and final partial batches.
- [ ] I understand `drop_last` and when not to use it.
- [ ] I can write a custom Dataset, `__len__`, and `__getitem__` from memory.
- [ ] I can apply preprocessing through a Dataset transform.
- [ ] I can create train and validation DataLoaders.
- [ ] I understand `num_workers`, `pin_memory`, and their limits.
- [ ] I understand `collate_fn` and can batch variable-length sequences.
- [ ] I can create dynamic padding and attention masks.
- [ ] I know where `.to(device)` belongs.
- [ ] I can identify common DataLoader bottlenecks.
- [ ] I can connect a DataLoader to a training loop.

You are ready when you can write the custom Dataset and dynamic collator without looking them up, predict batch shapes—including the partial final batch—and explain this line aloud:

```python
DataLoader(dataset, batch_size=32, shuffle=True, num_workers=4, pin_memory=True)
```